# Análise de Vendas - E-commerce DW

Este notebook realiza a análise exploratória dos dados de vendas do e-commerce.

In [ ]:
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns

## Conexão com o Banco de Dados

In [ ]:
# Configurar a conexão com o PostgreSQL
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "ecommerce"
DB_USER = "seu_usuario"
DB_PASSWORD = "sua_senha"

engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

## Carregar Dados

In [ ]:
# Carregar tabelas de análise
df_customers = pd.read_sql("SELECT * FROM dim_customers", engine)
df_products = pd.read_sql("SELECT * FROM dim_products", engine)
df_orders = pd.read_sql("SELECT * FROM fact_orders", engine)

print(f"Clientes: {len(df_customers)}")
print(f"Produtos: {len(df_products)}")
print(f"Pedidos: {len(df_orders)}")

## Visão Geral dos Dados

In [ ]:
df_orders.head()

In [ ]:
df_orders.describe()

## Análise de Vendas por Cliente

In [ ]:
# Top 10 clientes por valor gasto
top_customers = (
    df_customers.nlargest(10, "total_spent")
    [["first_name", "last_name", "total_orders", "total_spent"]]
)

plt.figure(figsize=(12, 6))
plt.barh(
    top_customers["first_name"] + " " + top_customers["last_name"],
    top_customers["total_spent"]
)
plt.xlabel("Valor Total Gasto")
plt.ylabel("Cliente")
plt.title("Top 10 Clientes por Valor Gasto")
plt.tight_layout()
plt.show()

## Análise de Vendas por Produto

In [ ]:
# Top 10 produtos por receita
top_products = df_products.nlargest(10, "total_revenue")

plt.figure(figsize=(12, 6))
plt.barh(top_products["product_name"], top_products["total_revenue"])
plt.xlabel("Receita Total")
plt.ylabel("Produto")
plt.title("Top 10 Produtos por Receita")
plt.tight_layout()
plt.show()

## Segmentação de Clientes

In [ ]:
# Distribuição de segmentos
segment_counts = df_customers["customer_segment"].value_counts()

plt.figure(figsize=(8, 6))
segment_counts.plot(kind="pie", autopct="%1.1f%%")
plt.title("Distribuição de Segmentos de Clientes")
plt.ylabel("")
plt.show()

## Análise Temporal

In [ ]:
# Vendas por mês
df_orders["order_date"] = pd.to_datetime(df_orders["order_date"])
monthly_sales = (
    df_orders.set_index("order_date")
    .resample("M")["total_price"]
    .sum()
)

plt.figure(figsize=(12, 6))
monthly_sales.plot()
plt.xlabel("Mês")
plt.ylabel("Vendas Totais")
plt.title("Vendas Mensais")
plt.tight_layout()
plt.show()